# Mapping ML articles to ML tasks (extended task descriptions)

The ML tasks were initially defined in the ML-Ontology (ml-ontology/ML_Ontology.ttl), and then given extended descriptions.

In [1]:
# Imports

import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

In [2]:
# Quick check of torch and cuda

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda:", torch.version.cuda)
print("device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0)) 


torch: 2.9.1+cu128
cuda available: True
torch cuda: 12.8
device count: 1
gpu name: NVIDIA A100-PCIE-40GB
capability: (8, 0)


In [5]:
ml_tasks_path = "../data/ml_tasks_extended.csv"
articles_path = "../data/ml_articles_dataset.csv"

model_name = "sentence-transformers/all-mpnet-base-v2"
TOP_K = 5
THRESHOLD = 0.35
BATCH_SIZE = 512


In [6]:
# Load data
tasks = pd.read_csv(ml_tasks_path)
articles = pd.read_csv(articles_path)

# Task profile : label + description
tasks["task_text"] = (tasks["taskLabel"].fillna("") + ". " + tasks["description"].fillna("")).str.strip()

# Document text : abstract + title
articles["doc_text"] = (articles["title"].fillna("") + ". " + articles["clean_abs"].fillna("")).str.strip()


In [7]:
tasks[["task", "taskLabel"]].head()

,task,taskLabel
0,http://h-da.de/ml-ontology/action_recognition,action recognition
1,http://h-da.de/ml-ontology/anomaly_detection,anomaly detection
2,http://h-da.de/ml-ontology/association_rule_le...,association rule learning
3,http://h-da.de/ml-ontology/audio_classification,audio classification
4,http://h-da.de/ml-ontology/audio_regression,audio regression


In [8]:
articles[["doi", "title", "doc_text"]].head()

,doi,title,doc_text
0,10.3390/asi6050076,Measuring Carbon in Cities and Their Buildings...,Measuring Carbon in Cities and Their Buildings...
1,10.1016/j.resconrec.2023.107073,Predictive modeling for the quantity of recycl...,Predictive modeling for the quantity of recycl...
2,10.30638/eemj.2023.018,END-OF-LIFE VEHICLES ASSESSMENT OF THE AUTOMOB...,END-OF-LIFE VEHICLES ASSESSMENT OF THE AUTOMOB...
3,10.1115/DETC2023-114718,PREDICTING THE QUANTITY OF RECYCLED END-OF-LIF...,PREDICTING THE QUANTITY OF RECYCLED END-OF-LIF...
4,10.1007/978-3-031-69626-8_78,Machine Learning Integration in LCA: Addressin...,Machine Learning Integration in LCA: Addressin...


In [9]:
# Load embedding model

model = SentenceTransformer(model_name)

# Tokenizer + max length
tokenizer = model.tokenizer
max_len = model.get_max_seq_length()

print("max length :", max_len)

max length : 384


In [10]:
# Embedding + normalization
def get_embedding(text: str):
    if not isinstance(text, str):
        text = ""
    return model.encode(
        text,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,  # dot-cosine
    )


In [11]:
# Chunking-embedding for large texts (above max length of 384)
def embed_with_chunking(text, max_len=max_len):
    # Handle non-string inputs
    if not isinstance(text, str):
        text = ""

    # Tokenize text
    tokens = tokenizer.tokenize(text)

    # Short enough -> embed directly
    if len(tokens) <= max_len:
        return get_embedding(text)

    # Divide into chunks
    chunks = [
        tokenizer.convert_tokens_to_string(tokens[i:i + max_len])
        for i in range(0, len(tokens), max_len)
    ]

    # Embed each chunk
    emb_chunks = [get_embedding(chunk) for chunk in chunks]

    # Average chunk embeddings (keep dimensions)
    return np.mean(emb_chunks, axis=0)


In [12]:
# Batch chunking-embedding (faster than one by one)
def embed_texts_chunked(texts, batch_size=BATCH_SIZE):
    embs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        for t in batch:
            embs.append(embed_with_chunking(t))
    return np.vstack(embs)


In [13]:
# Create embeddings for tasks and articles
task_emb = embed_texts_chunked(tasks["task_text"].tolist())
doc_emb  = embed_texts_chunked(articles["doc_text"].tolist())

task_emb.shape, doc_emb.shape


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (537 > 384). Running this sequence through the model will result in indexing errors


((40, 768), (15205, 768))

In [14]:
# Cosine similarity (dot product due to normalization) + top-k
S = doc_emb @ task_emb.T  # (n_docs, n_tasks)

topk_idx = np.argsort(-S, axis=1)[:, :TOP_K]
topk_scores = np.take_along_axis(S, topk_idx, axis=1)

topk_idx[:3], topk_scores[:3]


(array([[22,  9, 27,  4, 14],
        [22, 33,  9, 27,  4],
        [22,  5,  9, 27,  6]]),
 array([[0.3316713 , 0.28866306, 0.2492181 , 0.24356608, 0.23766664],
        [0.38762563, 0.3640241 , 0.3628295 , 0.34657574, 0.3353665 ],
        [0.33718944, 0.2767517 , 0.2712373 , 0.27070665, 0.25977626]],
       dtype=float32))

In [15]:
# Pack results into long-format table
rows = []
for di in range(S.shape[0]):
    doi = articles.loc[di, "doi"]
    for rank in range(TOP_K):
        ti = int(topk_idx[di, rank])
        score = float(topk_scores[di, rank])

        # Apply threshold (optional)
        if THRESHOLD is not None and score < THRESHOLD:
            continue

        rows.append({
            "doi": doi,
            "task": tasks.loc[ti, "task"],
            "taskLabel": tasks.loc[ti, "taskLabel"],
            "score": score,
            "rank": rank + 1,
        })

matches = pd.DataFrame(rows).sort_values(["doi", "rank"])
matches.head(15)


,doi,task,taskLabel,score,rank
5134,10.1002/9781118462706,http://h-da.de/ml-ontology/regression,regression,0.416487,1
5135,10.1002/9781118462706,http://h-da.de/ml-ontology/image_regression,image regression,0.375300,2
5136,10.1002/9781118462706,http://h-da.de/ml-ontology/audio_regression,audio regression,0.351332,3
16885,10.1002/9781119836780.ch7,http://h-da.de/ml-ontology/binary_classification,binary classification,0.355969,1
10513,10.1002/9781119847717.ch7,http://h-da.de/ml-ontology/time_series_classif...,time series classification,0.402593,1
10514,10.1002/9781119847717.ch7,http://h-da.de/ml-ontology/binary_classification,binary classification,0.367920,2
10515,10.1002/9781119847717.ch7,http://h-da.de/ml-ontology/audio_classification,audio classification,0.350535,3
17950,10.1002/9781119865605.ch3,http://h-da.de/ml-ontology/feature_extraction,feature extraction,0.386420,1
1933,10.1002/9781394155408,http://h-da.de/ml-ontology/association_rule_le...,association rule learning,0.489374,1
1934,10.1002/9781394155408,http://h-da.de/ml-ontology/feature_extraction,feature extraction,0.483775,2


In [16]:
# Select "best" task per article (rank=1) and save
best = matches[matches["rank"] == 1].copy()

best.to_csv("../results/extended_ml_task_mapping/task_matches_best.csv", index=False)
matches.to_csv("../results/extended_ml_task_mapping/task_matches_topk.csv", index=False)

print("Wrote: task_matches_best.csv, task_matches_topk.csv")
print("Best matches:", len(best), "of", len(articles))


Wrote: task_matches_best.csv, task_matches_topk.csv
Best matches: 8337 of 15205


In [17]:
# Find articles without a match (due to threshold) for manual review
matched_dois = set(best["doi"])
no_match = articles[~articles["doi"].isin(matched_dois)][["doi", "title"]].copy()

no_match.to_csv("../results/extended_ml_task_mapping/task_matches_no_match.csv", index=False)
print("No match:", len(no_match))
no_match.head(10)


No match: 6868


,doi,title
0,10.3390/asi6050076,Measuring Carbon in Cities and Their Buildings...
2,10.30638/eemj.2023.018,END-OF-LIFE VEHICLES ASSESSMENT OF THE AUTOMOB...
3,10.1115/DETC2023-114718,PREDICTING THE QUANTITY OF RECYCLED END-OF-LIF...
8,10.1002/amp2.70032,Improvements to Disassembly Lot Sizing With Ta...
10,10.1016/j.dche.2023.100103,Machine learning applications in biomass pyrol...
15,10.1109/ICMCSI64620.2025.10883185,Sustainable Industrial Symbiosis using IoT and...
17,10.1016/j.enconman.2024.118996,A systematic study involving patent analysis a...
21,10.1109/TII.2025.3598490,Computerized Automation and Machine Learning f...
22,10.1016/j.energy.2021.120113,Improving energy efficiency of carbon fiber ma...
26,10.1016/j.conbuildmat.2023.132502,Manufacture of artificial lightweight aggregat...
